# VTuber Builder · Colab 기본 화면 (Gradio 없음)


## ① 저장소 준비 (AI 다운로드 없음)


In [ ]:
import os, pathlib, shutil, signal, subprocess, sys, time

REPO_URL = "https://github.com/jujumelona/Virtual-pipeline.git"
REPO_DIR = pathlib.Path("/content/Virtual-pipeline")
SETUP_LOG = pathlib.Path("/content/vtuber_builder/logs/colab_bootstrap.log")

def run(command, timeout):
    """Stream every stdout/stderr line in Colab and save the identical transcript."""
    SETUP_LOG.parent.mkdir(parents=True, exist_ok=True)
    label = subprocess.list2cmdline(command)
    print("[준비] $", label, "· 전체 로그:", SETUP_LOG, flush=True)
    with SETUP_LOG.open("a", encoding="utf-8") as output:
        output.write("\n$ " + label + "\n")
        output.flush()
        start_offset = output.tell()
        process = subprocess.Popen(
            command, stdout=output, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    cursor = start_offset
    started = time.monotonic()

    def show_new_output():
        nonlocal cursor
        with SETUP_LOG.open("r", encoding="utf-8", errors="replace") as reader:
            reader.seek(cursor)
            while True:
                line = reader.readline()
                if not line:
                    break
                print(line, end="" if line.endswith("\n") else "\n", flush=True)
            cursor = reader.tell()

    try:
        while process.poll() is None:
            show_new_output()
            if time.monotonic() - started > timeout:
                raise TimeoutError(f"환경 준비 제한 시간 {timeout}초 초과 · {label}")
            time.sleep(0.1)
        show_new_output()
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                process.wait(timeout=10)
        show_new_output()
        raise
    if process.returncode:
        raise RuntimeError(
            f"환경 준비 실패(exit={process.returncode}): {label}\n"
            f"모든 stdout/stderr가 위에 출력되었습니다. 파일: {SETUP_LOG}"
        )

print("① 저장소 동기화")
if (REPO_DIR / ".git").is_dir():
    run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], 60)
    run(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", "main"], 180)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], 300)
run(["git", "-C", str(REPO_DIR), "checkout", "-B", "main", "origin/main"], 60)
run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], 60)

sys.path.insert(0, str(REPO_DIR))
from tools.colab_native import stop_legacy_server
stop_legacy_server()

print("① 저장소 연결 완료. ② 옵션 선택 후 ③ 다운로드 셀에서 프로그램과 모델을 병렬 준비하세요.", flush=True)


## ② 제작 옵션 선택


In [ ]:
#@title ② 작업 종류 선택 · 캐릭터 생성이 기본값
#@markdown **먼저 '캐릭터 생성' 또는 '액세서리 제작'을 선택하세요.**
TASK = "캐릭터 생성" #@param ["캐릭터 생성", "액세서리 제작"]
#@markdown **캐릭터 생성인 경우에만 적용**
MODE = "3d" #@param ["3d", "inochi2d", "live2d"]
USAGE = "corporation" #@param ["corporation", "personalProfit", "personalNonProfit"]
EXISTING_IMAGE_PATH = "" #@param {type:"string"}
MULTI_REFERENCE_3D = False #@param {type:"boolean"}
#@markdown **아래 두 옵션은 액세서리 제작을 선택했을 때만 적용**
#@markdown AUTO: 파일명으로 위치 지정 · ALL: 각 이미지에 모든 지원 위치 적용
ACCESSORY_ANCHOR = "AUTO" #@param ["AUTO", "ALL", "HEAD_TOP", "FACE", "LEFT_EAR", "RIGHT_EAR", "NECK", "CHEST", "BACK", "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HAND", "RIGHT_HAND", "LEFT_FOOT", "RIGHT_FOOT", "HIPS"]
ACCESSORY_BASE_VRM_PATH = "" #@param {type:"string"}
print(f"작업 종류: {TASK}")
if TASK == "캐릭터 생성":
    print(f"캐릭터 생성 방식: {MODE} · 사용 범위: {USAGE}")
else:
    print(f"액세서리 한 번에 생성 · 부착 범위: {ACCESSORY_ANCHOR}")
print("입력 이미지 경로가 비어 있으면 ③ 셀에서 Colab 파일 선택 창을 표시합니다.")


## ③ AI 모델·의존성·프로그램 다운로드


In [ ]:
#@title ③ AI 모델·의존성·필수 프로그램 다운로드 (독립 작업 병렬 실행)
#@markdown 선택한 제작 모드의 필수 프로그램과 AI 체크포인트를 설치·다운로드·검증합니다. 생성 작업은 하지 않습니다.
import sys, pathlib
if "TASK" not in globals():
    raise RuntimeError("② 제작 옵션 셀을 먼저 실행하세요.")
repo = pathlib.Path("/content/Virtual-pipeline")
if not (repo / "tools" / "colab_mode_prepare.py").is_file():
    raise RuntimeError("① 저장소 설치/동기화 셀을 먼저 실행하세요.")
scope = "3d" if TASK == "액세서리 제작" else MODE
if scope not in ("3d", "inochi2d", "live2d"):
    raise ValueError("지원하지 않는 모델 다운로드 모드: " + str(scope))
print(f"[다운로드] 선택 모드={scope} 사용 범위={USAGE}", flush=True)
# Full-body 3D starts with alpha segmentation before face detection; accessory
# fitting has another first GPU stage. Do not hold the unused face detector
# resident in either case.
prewarm_face = TASK == "캐릭터 생성" and not (MODE == "3d" and MULTI_REFERENCE_3D)
command = [sys.executable, "-u", str(repo / "tools" / "colab_mode_prepare.py"),
           "--mode", scope, "--usage", USAGE]
if not prewarm_face:
    command.append("--no-first-gpu-prewarm")
run(command, timeout=12000)
MODEL_DOWNLOAD_READY = scope
print("③ 필수 프로그램·의존성·모델 설치/검증 완료. ④ 사진 업로드 셀을 실행하세요.", flush=True)


## ④ 사진·VRM 업로드


In [ ]:
#@title ④ 사진·액세서리 업로드 전용 셀 (제작하지 않음)
#@markdown 업로드는 이 셀에서만 실행합니다. ⑤ ⑤ 제작 셀은 업로드 대화상자를 띄우지 않습니다.
from pathlib import Path
import sys, uuid
repo = Path("/content/Virtual-pipeline")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from tools.colab_native import _stored_uploads, _existing_image, _latest_avatar, _has_vrm_container
from google.colab import files

UPLOAD_ROOT = Path("/content/vtuber_builder/notebook_inputs") / ("manual_" + uuid.uuid4().hex)
INPUT_IMAGE_PATH = ""
REFERENCE_FACE_PATH = ""
UPLOADED_ACCESSORY_BASE = ""
UPLOADED_ACCESSORY_IMAGES = []

if TASK == "캐릭터 생성":
    if EXISTING_IMAGE_PATH.strip():
        INPUT_IMAGE_PATH = _existing_image(EXISTING_IMAGE_PATH.strip(), description="캐릭터 원본")
        print("[업로드] 기존 캐릭터 파일 사용:", INPUT_IMAGE_PATH, flush=True)
    else:
        print("[업로드] 캐릭터 원본 이미지 1장을 선택하세요.", flush=True)
        INPUT_IMAGE_PATH = _stored_uploads(
            files.upload, UPLOAD_ROOT / "character", images=True, multiple=False
        )[0]
    if MODE == "3d" and MULTI_REFERENCE_3D:
        print("[업로드] 얼굴 정면 확대 이미지 1장을 별도로 선택하세요.", flush=True)
        REFERENCE_FACE_PATH = _stored_uploads(
            files.upload, UPLOAD_ROOT / "face", images=True, multiple=False
        )[0]
    print("[업로드 완료] 캐릭터:", INPUT_IMAGE_PATH, flush=True)
elif TASK == "액세서리 제작":
    if ACCESSORY_BASE_VRM_PATH.strip():
        UPLOADED_ACCESSORY_BASE = str(Path(ACCESSORY_BASE_VRM_PATH.strip()).expanduser().resolve(strict=True))
        if not _has_vrm_container(Path(UPLOADED_ACCESSORY_BASE)):
            raise ValueError("기준 VRM 파일이 유효하지 않습니다.")
        print("[업로드] 기존 기준 VRM 사용:", UPLOADED_ACCESSORY_BASE, flush=True)
    elif not _latest_avatar():
        print("[업로드] 기준 VRM 파일 1개를 선택하세요.", flush=True)
        UPLOADED_ACCESSORY_BASE = _stored_uploads(
            files.upload, UPLOAD_ROOT / "base_vrm", images=False, multiple=False
        )[0]
        if not _has_vrm_container(Path(UPLOADED_ACCESSORY_BASE)):
            raise ValueError("업로드한 기준 VRM 파일이 유효하지 않습니다.")
    else:
        print("[업로드] 이전에 완성한 기준 VRM을 재사용합니다.", flush=True)
    print("[업로드] 액세서리 이미지 1~8장을 선택하세요.", flush=True)
    UPLOADED_ACCESSORY_IMAGES = _stored_uploads(
        files.upload, UPLOAD_ROOT / "accessories", images=True, multiple=True
    )
    if not 1 <= len(UPLOADED_ACCESSORY_IMAGES) <= 8:
        raise ValueError("액세서리 이미지는 1~8장이어야 합니다.")
    print("[업로드 완료] 액세서리 이미지:", len(UPLOADED_ACCESSORY_IMAGES), "장", flush=True)
else:
    raise ValueError("지원하지 않는 TASK: " + str(TASK))
print("업로드 셀이 끝났습니다. 다음 ⑤ 제작 셀을 수동 실행하세요.", flush=True)


## ⑤ 모델 제작


In [ ]:
#@title ⑤ 제작 전용 셀 (업로드·다운로드 없음)
#@markdown ④ 업로드가 끝난 뒤 실행합니다. 작업이 끝나기 전까지 이 셀을 실행 상태로 유지합니다.
import pathlib, sys, os
repo = pathlib.Path("/content/Virtual-pipeline")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from tools.colab_native import generate

# Generation must only consume already downloaded/verified model checkpoints.
# This flag is inherited by the isolated generation worker.
os.environ["VTUBER_NOTEBOOK_EXPLICIT_DOWNLOAD"] = "1"
if "MODEL_DOWNLOAD_READY" not in globals():
    raise RuntimeError("③ AI 모델·의존성·프로그램 다운로드 셀을 먼저 실행하세요.")
selected_scope = "3d" if TASK == "액세서리 제작" else MODE
if MODEL_DOWNLOAD_READY != selected_scope:
    raise RuntimeError("제작 모드를 변경했습니다. ③ 다운로드 셀을 다시 실행하세요.")
RESULT_FILE = None
if TASK == "캐릭터 생성":
    if not globals().get("INPUT_IMAGE_PATH") or not pathlib.Path(INPUT_IMAGE_PATH).is_file():
        raise RuntimeError("③ 업로드 셀에서 캐릭터 원본을 먼저 준비하세요.")
    if MODE == "3d" and MULTI_REFERENCE_3D and (
        not globals().get("REFERENCE_FACE_PATH")
        or not pathlib.Path(REFERENCE_FACE_PATH).is_file()
    ):
        raise RuntimeError("③ 업로드 셀에서 정면 얼굴 참조 이미지를 준비하세요.")
    RESULT_FILE = generate(
        MODE, USAGE, image_path=INPUT_IMAGE_PATH, full_body=MULTI_REFERENCE_3D,
        reference_face_path=globals().get("REFERENCE_FACE_PATH", ""),
        upload=lambda: (_ for _ in ()).throw(RuntimeError(
            "제작 중 업로드 호출 금지: ③ 업로드 셀을 먼저 실행하세요."
        )),
    )
elif TASK == "액세서리 제작":
    accessories = globals().get("UPLOADED_ACCESSORY_IMAGES", [])
    if not accessories or any(not pathlib.Path(x).is_file() for x in accessories):
        raise RuntimeError("③ 업로드 셀에서 액세서리 이미지를 먼저 준비하세요.")
    RESULT_FILE = generate(
        "accessory", USAGE, accessory_anchor=ACCESSORY_ANCHOR,
        accessory_base_path=globals().get("UPLOADED_ACCESSORY_BASE", ""),
        accessory_image_paths=tuple(accessories),
        upload=lambda: (_ for _ in ()).throw(RuntimeError(
            "제작 중 업로드 호출 금지: ③ 업로드 셀을 먼저 실행하세요."
        )),
    )
else:
    raise ValueError("작업 종류가 잘못됐습니다: " + str(TASK))

if not RESULT_FILE:
    raise RuntimeError(
        "제작 실패: 모델 작업이 완료되지 않았습니다. 위의 단계별 실패 원인과 "
        "/content/vtuber_builder/jobs/*/generation.log를 확인하세요."
    )
RESULT_FILE = str(pathlib.Path(RESULT_FILE).resolve())
if not pathlib.Path(RESULT_FILE).is_file():
    raise RuntimeError("제작 결과 파일이 존재하지 않습니다: " + RESULT_FILE)
print("제작 완료, 결과 파일:", RESULT_FILE, flush=True)
print("다운로드는 ⑥ 결과 다운로드 셀에서 따로 실행하세요.", flush=True)


## ⑥ 제작 결과 파일 다운로드


In [ ]:
#@title ⑥ 제작 결과 파일 다운로드 전용 셀 (완성 후 수동 실행)
#@markdown 제작이 성공했을 때만 다운로드를 누르세요.
DOWNLOAD_NOW = False #@param {type:"boolean"}
from pathlib import Path
if not DOWNLOAD_NOW:
    print("다운로드 대기 중. 다운로드하려면 DOWNLOAD_NOW를 켜고 ⑤ 셀만 다시 실행하세요.")
else:
    result = globals().get("RESULT_FILE")
    if not result or not Path(str(result)).is_file():
        raise RuntimeError("검증된 제작 결과가 없습니다. ⑤ 제작 셀을 먼저 완료하세요.")
    from google.colab import files
    print("다운로드 시작:", result, flush=True)
    files.download(str(result))


## ⑦ 상태 및 로그 진단


In [ ]:
#@title ⑦ 상태 진단 셀 · 실제 제작 상태 및 실패 로그 확인 (재연결 후에도 수동 실행 가능)
#@markdown 이 셀은 **모델 생성 완료를 자동으로 주장하지 않습니다.** 파일과 상태 기록을 검증합니다.
import json
from pathlib import Path

jobs = Path("/content/vtuber_builder/jobs")
reports = sorted(jobs.glob("*/status.json"), key=lambda x: x.stat().st_mtime, reverse=True) if jobs.is_dir() else []
result_file = globals().get("RESULT_FILE")
if reports:
    status_file = reports[0]
    data = json.loads(status_file.read_text(encoding="utf-8"))
    state = str(data.get("state", "unknown"))
    job_folder = status_file.parent
    print("최근 작업:", job_folder, flush=True)
    print("실제 작업 상태:", state, flush=True)
    print("worker PID:", data.get("pid"), "종료 코드:", data.get("exit_code", "기록 없음"), flush=True)
    print("전체 로그:", job_folder / "generation.log", flush=True)
    if state == "complete":
        if not result_file or not Path(str(result_file)).is_file():
            print("주의: 상태 기록은 complete이지만, 현재 세션의 결과 파일이 확인되지 않았습니다.", flush=True)
    elif state == "running":
        print("주의: running 기록만으로는 실제 프로세스가 살아 있다고 확인할 수 없습니다.", flush=True)
    else:
        print("제작 실패/취소 · 완성 모델로 취급하지 않습니다.", flush=True)
elif result_file and Path(str(result_file)).is_file():
    print("검증된 결과 파일:", Path(str(result_file)).resolve(), flush=True)
else:
    print("검증된 결과 파일이 없습니다. 작업 중 Colab 런타임이 삭제되면 /content 의 임시 로그도 사라질 수 있습니다.", flush=True)
print("Colab 오른쪽에 '런타임에 연결되어 있지 않습니다'가 표시된다면 이 셀도 실행할 수 없습니다.", flush=True)
print("먼저 Colab '다시 연결'을 시도해야 합니다. 무료 T4 할당량이 없다면 런타임이 재할당될 때까지 재실행할 수 없습니다.", flush=True)


## ⑧ 마지막 셀


In [ ]:
#@title ⑧ 마지막 셀 (작업 수행 없음)
print("마지막 셀: ③ 프로그램·모델 병렬 다운로드 / ④ 사진 업로드 / ⑤ 제작 / ⑥ 결과 파일 다운로드")
print("Colab이 연결을 끊은 경우 마지막 셀만으로 런타임을 유지할 수는 없습니다.")
